In [1]:
import pandas as pd
import json

In [2]:
df=pd.read_csv("SampleSuperstore.csv")

In [3]:
df.head(2)

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.96,2,0.0,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.94,3,0.0,219.5820


In [4]:
df["Ship Mode"].value_counts()

Ship Mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64

In [5]:
df["Region"].value_counts()

Region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64

In [6]:
df["Sales"].value_counts()

Sales
12.960     56
19.440     39
15.552     39
25.920     36
10.368     36
           ..
4.240       1
319.960     1
646.740     1
81.940      1
243.160     1
Name: count, Length: 5825, dtype: int64

In [7]:
df.dtypes

Ship Mode        object
Segment          object
Country          object
City             object
State            object
Postal Code       int64
Region           object
Category         object
Sub-Category     object
Sales           float64
Quantity          int64
Discount        float64
Profit          float64
dtype: object

In [8]:
df["Discount"].value_counts()

Discount
0.00    4798
0.20    3657
0.70     418
0.80     300
0.30     227
0.40     206
0.60     138
0.10      94
0.50      66
0.15      52
0.32      27
0.45      11
Name: count, dtype: int64

In [9]:
df["Discount"]=(df["Discount"]*100).astype(int)

In [10]:
df["Discount"].value_counts()

Discount
0     4798
20    3657
70     418
80     300
30     227
40     206
60     138
10      94
50      66
15      52
32      27
45      11
Name: count, dtype: int64

In [11]:
df["City"]=df["City"].replace("New York City","New York")

In [12]:
df["Category"].value_counts()

Category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64

In [13]:
df["Sub-Category"].value_counts()

Sub-Category
Binders        1523
Paper          1370
Furnishings     957
Phones          889
Storage         846
Art             796
Accessories     775
Chairs          617
Appliances      466
Labels          364
Tables          319
Envelopes       254
Bookcases       228
Fasteners       217
Supplies        190
Machines        115
Copiers          68
Name: count, dtype: int64

In [ ]:
def agg_to_chunk(df_agg: pd.DataFrame,
                 group_keys: list[str],
                 granularity: str,
                 text_template_fn) -> list[dict]:
    chunks = []
    for _, row in df_agg.iterrows():
        text = text_template_fn(row)
        metadata = {k: row[k] for k in group_keys}
        metadata["granularity"] = granularity
        metadata["sales_total"] = round(row["Sales"], 2)
        metadata["profit_total"] = round(row["Profit"], 2)
        metadata["quantity_total"] = int(row["Quantity"])
        metadata["avg_discount"] = round(row["Discount"], 4)
        metadata["row_count"] = int(row["row_count"])
        chunks.append({"text": text, "metadata": metadata})
    return chunks


def chunk_overall_summary(df: pd.DataFrame) -> list[dict]:
    total_sales = df["Sales"].sum()
    total_profit = df["Profit"].sum()
    total_qty = df["Quantity"].sum()
    avg_discount = df["Discount"].mean()
    top_region = df.groupby("Region")["Sales"].sum().idxmax()
    top_category = df.groupby("Category")["Sales"].sum().idxmax()
    top_segment = df.groupby("Segment")["Sales"].sum().idxmax()
    top_ship_mode = df.groupby("Ship Mode")["Sales"].sum().idxmax()
    profit_margin = total_profit / total_sales * 100

    text = (
        f"Overall dataset summary: Total Sales = ${total_sales:,.2f}, "
        f"Total Profit = ${total_profit:,.2f} (margin {profit_margin:.1f}%), "
        f"Total Quantity = {total_qty:,}, "
        f"Average Discount = {avg_discount:.1%}. "
        f"Top performing Region: {top_region}. "
        f"Top Category by Sales: {top_category}. "
        f"Top Customer Segment: {top_segment}. "
        f"Most used Ship Mode: {top_ship_mode}."
    )
    return [{"text": text, "metadata": {"granularity": "overall"}}]


def chunk_by_region(df: pd.DataFrame) -> list[dict]:
    agg = (
        df.groupby("Region")
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
             Quantity=("Quantity", "sum"), Discount=("Discount", "mean"),
             row_count=("Sales", "count"))
        .reset_index()
    )
    def tmpl(row):
        margin = row["Profit"] / row["Sales"] * 100
        return (
            f"Region '{row['Region']}': Sales = ${row['Sales']:,.2f}, "
            f"Profit = ${row['Profit']:,.2f} (margin {margin:.1f}%), "
            f"Quantity = {int(row['Quantity']):,}, "
            f"Avg Discount = {row['Discount']:.1%}, "
            f"based on {int(row['row_count'])} orders."
        )
    return agg_to_chunk(agg, ["Region"], "region", tmpl)


def chunk_by_category(df: pd.DataFrame) -> list[dict]:
    agg = (
        df.groupby("Category")
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
             Quantity=("Quantity", "sum"), Discount=("Discount", "mean"),
             row_count=("Sales", "count"))
        .reset_index()
    )
    def tmpl(row):
        margin = row["Profit"] / row["Sales"] * 100
        return (
            f"Category '{row['Category']}': Sales = ${row['Sales']:,.2f}, "
            f"Profit = ${row['Profit']:,.2f} (margin {margin:.1f}%), "
            f"Quantity = {int(row['Quantity']):,}, "
            f"Avg Discount = {row['Discount']:.1%}."
        )
    return agg_to_chunk(agg, ["Category"], "category", tmpl)


def chunk_by_segment(df: pd.DataFrame) -> list[dict]:
    agg = (
        df.groupby("Segment")
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
             Quantity=("Quantity", "sum"), Discount=("Discount", "mean"),
             row_count=("Sales", "count"))
        .reset_index()
    )
    def tmpl(row):
        return (
            f"Customer Segment '{row['Segment']}': Sales = ${row['Sales']:,.2f}, "
            f"Profit = ${row['Profit']:,.2f}, "
            f"Quantity = {int(row['Quantity']):,}, "
            f"Avg Discount = {row['Discount']:.1%}."
        )
    return agg_to_chunk(agg, ["Segment"], "segment", tmpl)


def chunk_by_ship_mode(df: pd.DataFrame) -> list[dict]:
    agg = (
        df.groupby("Ship Mode")
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
             Quantity=("Quantity", "sum"), Discount=("Discount", "mean"),
             row_count=("Sales", "count"))
        .reset_index()
    )
    def tmpl(row):
        return (
            f"Ship Mode '{row['Ship Mode']}': Sales = ${row['Sales']:,.2f}, "
            f"Profit = ${row['Profit']:,.2f}, "
            f"Quantity = {int(row['Quantity']):,}, "
            f"number of orders = {int(row['row_count'])}."
        )
    return agg_to_chunk(agg, ["Ship Mode"], "ship_mode", tmpl)


def chunk_by_region_category(df: pd.DataFrame) -> list[dict]:
    agg = (
        df.groupby(["Region", "Category"])
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
             Quantity=("Quantity", "sum"), Discount=("Discount", "mean"),
             row_count=("Sales", "count"))
        .reset_index()
    )
    def tmpl(row):
        margin = row["Profit"] / row["Sales"] * 100
        return (
            f"In the {row['Region']} region, Category '{row['Category']}': Sales = ${row['Sales']:,.2f}, "
            f"Profit = ${row['Profit']:,.2f} (margin {margin:.1f}%), "
            f"Quantity = {int(row['Quantity']):,}, "
            f"Avg Discount = {row['Discount']:.1%}."
        )
    return agg_to_chunk(agg, ["Region", "Category"], "region_category", tmpl)


def chunk_by_segment_subcategory(df: pd.DataFrame) -> list[dict]:
    agg = (
        df.groupby(["Segment", "Sub-Category"])
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
             Quantity=("Quantity", "sum"), Discount=("Discount", "mean"),
             row_count=("Sales", "count"))
        .reset_index()
    )
    def tmpl(row):
        return (
            f"Customer Segment '{row['Segment']}', Sub-Category '{row['Sub-Category']}': Sales = ${row['Sales']:,.2f}, "
            f"Profit = ${row['Profit']:,.2f}, "
            f"Quantity = {int(row['Quantity']):,}, "
            f"Avg Discount = {row['Discount']:.1%}."
        )
    return agg_to_chunk(agg, ["Segment", "Sub-Category"], "segment_subcategory", tmpl)


def chunk_by_state(df: pd.DataFrame) -> list[dict]:
    agg = (
        df.groupby(["State", "Region"])
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
             Quantity=("Quantity", "sum"), Discount=("Discount", "mean"),
             row_count=("Sales", "count"))
        .reset_index()
    )
    def tmpl(row):
        margin = row["Profit"] / row["Sales"] * 100
        return (
            f"State '{row['State']}' (Region: {row['Region']}): Sales = ${row['Sales']:,.2f}, "
            f"Profit = ${row['Profit']:,.2f} (margin {margin:.1f}%), "
            f"Quantity = {int(row['Quantity']):,}."
        )
    return agg_to_chunk(agg, ["State", "Region"], "state", tmpl)


def chunk_discount_impact(df: pd.DataFrame) -> list[dict]:
    df2 = df.copy()
    df2["discount_bracket"] = pd.cut(df2["Discount"])

    agg = (
        df2.groupby("discount_bracket", observed=True)
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
             Quantity=("Quantity", "sum"), row_count=("Sales", "count"))
        .reset_index()
    )
    chunks = []
    for _, row in agg.iterrows():
        text = (
            f"Orders with discount bracket '{row['discount_bracket']}': Sales = ${row['Sales']:,.2f}, "
            f"Profit = ${row['Profit']:,.2f}, "
            f"Quantity = {int(row['Quantity']):,}, "
            f"Number of orders = {int(row['row_count'])}."
        )
        chunks.append({
            "text": text,
            "metadata": {
                "granularity": "discount_impact",
                "discount_bracket": str(row["discount_bracket"]),
                "sales_total": round(row["Sales"], 2),
                "profit_total": round(row["Profit"], 2),
            }
        })
    return chunks


def build_all_chunks(df: pd.DataFrame) -> list[dict]:
    all_chunks = []
    strategies = [
        chunk_overall_summary,
        chunk_by_region,
        chunk_by_category,
        chunk_by_segment,
        chunk_by_ship_mode,
        chunk_by_region_category,
        chunk_by_segment_subcategory,
        chunk_by_state,
        chunk_discount_impact,
    ]
    for fn in strategies:
        all_chunks.extend(fn(df))

    for i, chunk in enumerate(all_chunks):
        chunk["chunk_id"] = f"chunk_{i:04d}"
    print(f"✅ Total chunks created: {len(all_chunks)}")
    return all_chunks


df_norm = df
documents = build_all_chunks(df_norm)
print(f"Loaded df with {len(df_norm):,} rows and built {len(documents)} chunks.")


✅ Total chunks created: 128
Loaded df with 9,994 rows and built 128 chunks.


In [16]:
df

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,Second Class,Consumer,United States,Miami,Florida,33180,South,Furniture,Furnishings,25.2480,3,20,4.1028
9990,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Furniture,Furnishings,91.9600,2,0,15.6332
9991,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Technology,Phones,258.5760,2,20,19.3932
9992,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Office Supplies,Paper,29.6000,4,0,13.3200


In [17]:
import json

# Save documents to JSON file for RAG system
output_file = "documents.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"✓ Documents saved to {output_file}")
print(f"Total documents: {len(documents)}")

# Display document structure
print("\nDocument structure by level:")
for doc in documents[:3]:  # Show first 3 as examples
    print(f"  {json.dumps(doc, ensure_ascii=False)[:100]}...")


✓ Documents saved to documents.json
Total documents: 128

Document structure by level:
  {"text": "Overall dataset summary: Total Sales = $2,297,200.86, Total Profit = $286,397.02 (margin 1...
  {"text": "Region 'Central': Sales = $501,239.89, Profit = $39,706.36 (margin 7.9%), Quantity = 8,780...
  {"text": "Region 'East': Sales = $678,781.24, Profit = $91,522.78 (margin 13.5%), Quantity = 10,618,...


In [25]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer('all-MiniLM-L6-v2')
text=[doc["text"] for doc in documents]
embeddings=model.encode(text,show_progress_bar=True)
print(f"Generated embeddings for {len(embeddings)} documents. Each embedding has dimension {len(embeddings[0])}.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6061.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 4/4 [00:00<00:00,  5.84it/s]

Generated embeddings for 128 documents. Each embedding has dimension 384.


In [27]:
import faiss
import numpy as np
import pickle
embeddings = np.array(embeddings).astype("float32")
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
metadatas = [doc["metadata"] for doc in documents]
# Add vectors
index.add(embeddings)


# ===== Save FAISS index =====
faiss.write_index(index, "faiss_index.bin")

# ===== Save metadata =====
with open("metadata.pkl", "wb") as f:
    pickle.dump({
        "texts": text,
        "metadatas": metadatas
    }, f)
print("✓ FAISS index and metadata saved.")

✓ FAISS index and metadata saved.


In [2]:
# Test loading FAISS index and metadata
import faiss
import pickle
import numpy as np

# Load FAISS index
index = faiss.read_index("faiss_index.bin")
print(f"FAISS index loaded. Number of vectors: {index.ntotal}")

# Load metadata
with open("metadata.pkl", "rb") as f:
    data = pickle.load(f)

texts = data["texts"]
metadatas = data["metadatas"]
print(f"Metadata loaded. Number of texts: {len(texts)}, metadatas: {len(metadatas)}")

# Test retrieval
query = "What is the total sales in the South region?"
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
query_embedding = model.encode([query]).astype("float32")

# Search
k = 5
distances, indices = index.search(query_embedding, k)

print(f"Top {k} similar chunks for query: '{query}'")
for i, idx in enumerate(indices[0]):
    print(f"{i+1}. Distance: {distances[0][i]:.4f}")
    print(f"   Text: {texts[idx][:100]}...")
    print(f"   Metadata: {metadatas[idx]}")
    print()

FAISS index loaded. Number of vectors: 128
Metadata loaded. Number of texts: 128, metadatas: 128


C:\Users\nguye\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5149.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top 5 similar chunks for query: 'What is the total sales in the South region?'
1. Distance: 0.4712
   Text: Region 'South': Sales = $391,721.91, Profit = $46,749.43 (margin 11.9%), Quantity = 6,209, Avg Disco...
   Metadata: {'Region': 'South', 'granularity': 'region', 'sales_total': 391721.91, 'profit_total': 46749.43, 'quantity_total': 6209, 'avg_discount': 14.7253, 'row_count': 1620}

2. Distance: 0.6652
   Text: State 'South Carolina' (Region: South): Sales = $8,481.71, Profit = $1,769.06 (margin 20.9%), Quanti...
   Metadata: {'State': 'South Carolina', 'Region': 'South', 'granularity': 'state', 'sales_total': 8481.71, 'profit_total': 1769.06, 'quantity_total': 172, 'avg_discount': 0.0, 'row_count': 42}

3. Distance: 0.6953
   Text: Region 'East': Sales = $678,781.24, Profit = $91,522.78 (margin 13.5%), Quantity = 10,618, Avg Disco...
   Metadata: {'Region': 'East', 'granularity': 'region', 'sales_total': 678781.24, 'profit_total': 91522.78, 'quantity_total': 10618, 'avg_discount'

In [3]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

class RAGRetriever:
    def __init__(self, faiss_index_path="faiss_index.bin", metadata_path="metadata.pkl", model_name='all-MiniLM-L6-v2'):
        # Load FAISS index
        self.index = faiss.read_index(faiss_index_path)
        print(f"✅ FAISS index loaded: {self.index.ntotal} vectors")

        # Load metadata
        with open(metadata_path, "rb") as f:
            data = pickle.load(f)
        self.texts = data["texts"]
        self.metadatas = data["metadatas"]
        print(f"✅ Metadata loaded: {len(self.texts)} texts, {len(self.metadatas)} metadatas")

        # Load embedding model
        self.model = SentenceTransformer(model_name)
        print(f"✅ Embedding model loaded: {model_name}")

    def retrieve(self, query, top_k=5):
        # Encode query
        query_embedding = self.model.encode([query]).astype("float32")

        # Search FAISS
        distances, indices = self.index.search(query_embedding, top_k)

        # Prepare results
        results = []
        for i, idx in enumerate(indices[0]):
            if idx < len(self.texts):  # Safety check
                result = {
                    "text": self.texts[idx],
                    "metadata": self.metadatas[idx],
                    "distance": float(distances[0][i]),
                    "rank": i + 1
                }
                results.append(result)

        return results

# Initialize retriever
retriever = RAGRetriever()

# Test retrieval
query = "What are the sales in the South region?"
results = retriever.retrieve(query, top_k=3)

print(f"\n🔍 Query: {query}")
print(f"📊 Top {len(results)} results:")
for res in results:
    print(f"\nRank {res['rank']} (Distance: {res['distance']:.4f})")
    print(f"Text: {res['text'][:100]}...")
    print(f"Granularity: {res['metadata'].get('granularity', 'N/A')}")
    if 'sales_total' in res['metadata']:
        print(f"Sales: ${res['metadata']['sales_total']:,.2f}")

✅ FAISS index loaded: 128 vectors
✅ Metadata loaded: 128 texts, 128 metadatas


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7923.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: all-MiniLM-L6-v2

🔍 Query: What are the sales in the South region?
📊 Top 3 results:

Rank 1 (Distance: 0.5391)
Text: Region 'South': Sales = $391,721.91, Profit = $46,749.43 (margin 11.9%), Quantity = 6,209, Avg Disco...
Granularity: region
Sales: $391,721.91

Rank 2 (Distance: 0.7555)
Text: State 'South Carolina' (Region: South): Sales = $8,481.71, Profit = $1,769.06 (margin 20.9%), Quanti...
Granularity: state
Sales: $8,481.71

Rank 3 (Distance: 0.7867)
Text: Region 'East': Sales = $678,781.24, Profit = $91,522.78 (margin 13.5%), Quantity = 10,618, Avg Disco...
Granularity: region
Sales: $678,781.24


In [4]:
# Test the separate retriever.py file
from retriever import RAGRetriever

# Initialize retriever from separate file
retriever_pro = RAGRetriever()

# Test retrieval with sorting by distance descending
query = "What are the sales in the South region?"
results = retriever_pro.retrieve(query, top_k=5)

print(f"\n🔍 Query: {query}")
print("📊 Results sorted by distance DESCENDING (worst to best similarity):")
for res in results:
    print(f"\nRank {res['rank']} (Distance: {res['distance']:.4f})")
    print(f"Granularity: {res['metadata'].get('granularity', 'N/A')}")
    if 'sales_total' in res['metadata']:
        print(f"Sales: ${res['metadata']['sales_total']:,.2f}")
    print(f"Text: {res['text'][:100]}...")
    print("-" * 50)

✅ FAISS index loaded: 128 vectors
✅ Metadata loaded: 128 texts, 128 metadatas


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2395.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: all-MiniLM-L6-v2

🔍 Query: What are the sales in the South region?
📊 Results sorted by distance DESCENDING (worst to best similarity):

Rank 5 (Distance: 0.8036)
Granularity: state
Sales: $1,315.56
Text: State 'South Dakota' (Region: Central): Sales = $1,315.56, Profit = $394.83 (margin 30.0%), Quantity...
--------------------------------------------------

Rank 4 (Distance: 0.8023)
Granularity: region_category
Sales: $148,771.91
Text: In the South region, Category 'Technology': Sales = $148,771.91, Profit = $19,991.83 (margin 13.4%),...
--------------------------------------------------

Rank 3 (Distance: 0.7867)
Granularity: region
Sales: $678,781.24
Text: Region 'East': Sales = $678,781.24, Profit = $91,522.78 (margin 13.5%), Quantity = 10,618, Avg Disco...
--------------------------------------------------

Rank 2 (Distance: 0.7555)
Granularity: state
Sales: $8,481.71
Text: State 'South Carolina' (Region: South): Sales = $8,481.71, Profit = $1,769.06 (marg

In [7]:
# Test Fairness-Aware Reranking
import importlib
import retriever
importlib.reload(retriever)
from retriever import RAGRetriever

# Initialize retriever
retriever = RAGRetriever()

# Test fairness-aware retrieval
query = "What are the sales in the South region?"

print("🧪 Testing Fairness-Aware Reranking")
print("="*80)

# Standard retrieval
standard_results = retriever.retrieve(query, top_k=5)
print(f"\n📊 Standard Retrieval (Top 5):")
retriever.print_results(standard_results)

# Fairness-aware retrieval
fairness_results = retriever.retrieve_with_fairness(query, top_k=5, candidate_pool_size=20)
print(f"\n🎯 Fairness-Aware Retrieval (Top 5 from 20 candidates):")
retriever.print_results(fairness_results)

# Compare diversity
print(f"\n📈 Diversity Comparison:")
from collections import Counter
standard_regions = [r['metadata'].get('Region', 'Unknown') for r in standard_results]
fairness_regions = [r['metadata'].get('Region', 'Unknown') for r in fairness_results]

print(f"Standard regions: {dict(Counter(standard_regions))}")
print(f"Fairness regions: {dict(Counter(fairness_regions))}")

# Test pre-filtering
print(f"\n🔍 Testing Pre-filtering (South region only):")
pre_filtered = retriever.pre_filter_by_metadata(query, {"Region": "South"}, top_k=5)
retriever.print_results(pre_filtered, "South region sales")

# Test post-filtering
print(f"\n🔍 Testing Post-filtering (diversity check):")
all_results = retriever.retrieve("sales data", top_k=10)
post_filtered = retriever.post_filter_diversity_check(all_results, max_region_ratio=0.5)
retriever.print_results(post_filtered, "sales data (diversity filtered)")

✅ FAISS index loaded: 128 vectors
✅ Metadata loaded: 128 texts, 128 metadatas


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7922.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: all-MiniLM-L6-v2
🧪 Testing Fairness-Aware Reranking

📊 Standard Retrieval (Top 5):
📊 Top 5 results (sorted by distance descending - worst to best similarity):

Rank 5 (Distance: 0.8036)
Granularity: state
Sales: $1,315.56
Profit: $394.83
Quantity: 42
Avg Discount: 0.00%
Text: State 'South Dakota' (Region: Central): Sales = $1,315.56, Profit = $394.83 (margin 30.0%), Quantity = 42....
----------------------------------------

Rank 4 (Distance: 0.8023)
Granularity: region_category
Sales: $148,771.91
Profit: $19,991.83
Quantity: 1,118
Avg Discount: 1078.50%
Text: In the South region, Category 'Technology': Sales = $148,771.91, Profit = $19,991.83 (margin 13.4%), Quantity = 1,118, Avg Discount = 1078.5%....
----------------------------------------

Rank 3 (Distance: 0.7867)
Granularity: region
Sales: $678,781.24
Profit: $91,522.78
Quantity: 10,618
Avg Discount: 1453.65%
Text: Region 'East': Sales = $678,781.24, Profit = $91,522.78 (margin 13.5%), Quantity = 10,618

In [1]:
# Test Google Gemini RAG System
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
print(f"GOOGLE_API_KEY loaded: {os.environ.get('GOOGLE_API_KEY') is not None}")
print(f"Current working directory: {os.getcwd()}")

# Test API key directly
import google.generativeai as genai
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

try:
    # List available models
    models = genai.list_models()
    print(f"Available models: {[m.name for m in models[:5]]}...")  # Show first 5
except Exception as e:
    print(f"API key test failed: {e}")

from retriever import RAGRetriever
from rag_llm import RAGWithLLM

# Initialize retriever
retriever = RAGRetriever(
    faiss_index_path="faiss_index.bin",
    metadata_path="metadata.pkl",
    model_name="all-MiniLM-L6-v2"
)

# Initialize RAG with Google Gemini
rag_system = RAGWithLLM(retriever, model_name="gemini-2.5-flash")

# Test query
test_query = "What are the total sales in the South region?"
print(f"\n🔍 Testing Query: {test_query}")

try:
    result = rag_system.query(test_query)
    print(f"✅ Success! Answer: {result['answer']}")
except Exception as e:
    print(f"❌ Error: {e}")

    print("\n🔧 Troubleshooting:")
    print("1. Check if GOOGLE_API_KEY is set in .env file")
    print("2. Verify API key is valid at: https://makersuite.google.com/app/apikey")
    print("3. Make sure packages are installed: pip install google-generativeai python-dotenv")

GOOGLE_API_KEY loaded: True
Current working directory: d:\vscode\Project\RAG(e-commerce)


c:\Users\nguye\anaconda3\envs\python3\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\nguye\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\nguye\AppData\Local\Temp\ipykernel_24580\2562623057.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for mo

API key test failed: 'generator' object is not subscriptable
✅ Loaded environment variables from .env
✅ FAISS index loaded: 128 vectors
✅ Metadata loaded: 128 texts, 128 metadatas


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7923.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: all-MiniLM-L6-v2
✅ Google Gemini initialized: gemini-2.5-flash

🔍 Testing Query: What are the total sales in the South region?
✅ Success! Answer: Based on the provided data:

**Total Sales in the South Region**

The total sales in the South region are **$391,721.91**.

This information is directly provided in data point [1]:
*   **Level:** region
*   **Region:** South
*   **Sales:** $391,721.91
*   **Profit:** $46,749.43
